<a href="https://colab.research.google.com/github/dcdlima/RAG_Tests/blob/main/Pipeline_RAG_GC_2026_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. Remove qualquer versão corrompida ou conflitante
%pip uninstall -y langchain langchain-community langchain-core langchain-openai langchain-chroma langchain-text-splitters

# 2. Instala os pacotes garantindo a versão mais recente e compatível
%pip install -qU langchain langchain-community langchain-core langchain-openai langchain-chroma langchain-text-splitters pypdf pandas

Found existing installation: langchain 1.3.1
Uninstalling langchain-1.3.1:
  Successfully uninstalled langchain-1.3.1
Found existing installation: langchain-community 0.4.2
Uninstalling langchain-community-0.4.2:
  Successfully uninstalled langchain-community-0.4.2
Found existing installation: langchain-core 1.4.0
Uninstalling langchain-core-1.4.0:
  Successfully uninstalled langchain-core-1.4.0
Found existing installation: langchain-openai 1.2.2
Uninstalling langchain-openai-1.2.2:
  Successfully uninstalled langchain-openai-1.2.2
Found existing installation: langchain-chroma 1.1.0
Uninstalling langchain-chroma-1.1.0:
  Successfully uninstalled langchain-chroma-1.1.0
Found existing installation: langchain-text-splitters 1.1.2
Uninstalling langchain-text-splitters-1.1.2:
  Successfully uninstalled langchain-text-splitters-1.1.2


In [3]:
import os
import glob
import pandas as pd

# Importações principais
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import PromptTemplate

# Importações LCEL Puro (Substituem o módulo 'chains')
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Configurações Iniciais
os.environ["OPENAI_API_KEY"] = "sk-sua-chave-aqui" # Insira sua chave
PASTA_PDFS = "/content/artigos"

# Expansão do léxico
PALAVRAS_CHAVE = [
    "prospecção tecnológica", "future forecast", "future events", "tech foresight",
    "tendências tecnológicas", "previsão de cenários", "estudos de futuros",
    "visão de futuro", "roadmapping tecnológico", "tecnologias emergentes",
    "horizonte tecnológico", "foresight"
]

# 2. Definição do Prompt
prompt_template = """
Você é um engenheiro de machine learning especialista em prospecção tecnológica.
Sua tarefa é analisar o texto extraído de um artigo científico e identificar a presença de eventos futuros, tendências, previsões tecnológicas ou cenários prospectivos.

Instruções cruciais:
1. Responda baseando-se EXCLUSIVAMENTE no contexto fornecido abaixo.
2. Se o contexto não contiver informações sobre o futuro ou tecnologias emergentes, responda exatamente: "Nenhum evento futuro ou tendência identificado no texto."
3. Caso encontre, resuma as descobertas de forma analítica e objetiva em um parágrafo.

Contexto do artigo:
{context}

Pergunta: {input}
Resposta:"""

PROMPT = PromptTemplate.from_template(prompt_template)

# Função auxiliar para extrair texto dos documentos recuperados
def formatar_documentos(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def analisar_artigos_prospecao(caminho_pasta: str) -> pd.DataFrame:
    arquivos_pdf = glob.glob(os.path.join(caminho_pasta, "*.pdf"))

    if not arquivos_pdf:
        print("Nenhum arquivo PDF encontrado na pasta.")
        return pd.DataFrame()

    resultados = []

    # Modelos
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    query_busca = f"Identifique informações sobre: {', '.join(PALAVRAS_CHAVE)}."

    for caminho_arquivo in arquivos_pdf:
        nome_arquivo = os.path.basename(caminho_arquivo)
        print(f"Processando: {nome_arquivo}...")

        try:
            # 3. Ingestão e Processamento
            loader = PyPDFLoader(caminho_arquivo)
            documentos = loader.load()

            text_splitter = RecursiveCharacterTextSplitter(
                chunk_size=1200,
                chunk_overlap=200,
                separators=["\n\n", "\n", ".", " "]
            )
            chunks = text_splitter.split_documents(documentos)

            # 4. Vetorização Efêmera
            vectorstore = Chroma.from_documents(
                documents=chunks,
                embedding=embeddings
            )

            # 5. Pipeline RAG com LCEL PURO (Bypass no langchain.chains)
            retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

            rag_chain = (
                {"context": retriever | formatar_documentos, "input": RunnablePassthrough()}
                | PROMPT
                | llm
                | StrOutputParser()
            )

            # 6. Execução
            texto_gerado = rag_chain.invoke(query_busca)

            resultados.append({
                "Arquivo": nome_arquivo,
                "Eventos Futuros e Tendências": texto_gerado
            })

            # Limpeza do BD Vetorial
            vectorstore.delete_collection()

        except Exception as e:
            print(f"Erro ao processar {nome_arquivo}: {e}")
            resultados.append({
                "Arquivo": nome_arquivo,
                "Eventos Futuros e Tendências": f"Erro de processamento: {e}"
            })

    return pd.DataFrame(resultados)

# Execução
if __name__ == "__main__":
    tabela_final = analisar_artigos_prospecao(PASTA_PDFS)
    display(tabela_final)

Processando: Mapping-the-evolution-of-GPT4o-innovations-An-analytical-exploration-of-progress-applications-and-horizons_2025_Universidad-Nacional-de-San-Martin (2).pdf...
Erro ao processar Mapping-the-evolution-of-GPT4o-innovations-An-analytical-exploration-of-progress-applications-and-horizons_2025_Universidad-Nacional-de-San-Martin (2).pdf: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-sua-c*****aqui. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}


,Arquivo,Eventos Futuros e Tendências
0,Mapping-the-evolution-of-GPT4o-innovations-An-...,Erro de processamento: Error code: 401 - {'err...
